In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

In [ ]:
# prompt: read in a .csv file called sound_pollutant_met_data.csv, then show the first few rows of the data

import pandas as pd

# Assuming the file is in the current working directory.
# If not, provide the full path to the file.
try:
    df = pd.read_csv('sound_pollutant_met_data.csv')
    print(df.head())
except FileNotFoundError:
    print("Error: 'sound_pollutant_met_data.csv' not found. Please make sure the file exists in the current directory or provide the correct path.")



In [ ]:
# Renaming columns by replacing spaces with underscores
df.rename(columns=lambda col: col.replace(" ", "_"), inplace=True)

# Output the updated column names
print(df.columns)


In [ ]:
print(df.columns.tolist())

In [ ]:


# Define predictors and response
predictors = [
    "LAFmax", "LAFmin", "LCpeak", "LAeq", "LCeq", "LZeq_12.5Hz",
    "LZeq_16Hz", "LZeq_20Hz", "LZeq_25Hz", "LZeq_31.5Hz", "LZeq_40Hz",
    "LZeq_50Hz", "LZeq_63Hz", "LZeq_80Hz", "LZeq_100Hz", "LZeq_125Hz",
    "LZeq_160Hz", "LZeq_200Hz", "LZeq_250Hz", "LZeq_315Hz", "LZeq_400Hz",
    "LZeq_500Hz", "LZeq_630Hz", "LZeq_800Hz", "LZeq_1kHz", "LZeq_1.25kHz",
    "LZeq_1.6kHz", "LZeq_2kHz", "LZeq_2.5kHz", "LZeq_3.15kHz", "LZeq_4kHz",
    "LZeq_5kHz", "LZeq_6.3kHz", "LZeq_8kHz", "LZeq_10kHz", "LZeq_12.5kHz",
    "LZeq_16kHz"
]
response = "nox"

# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(df[col].dtype, np.floating):
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values after conversion
df = df.dropna(subset=predictors + [response])

# Step 2: Split the data
train_data, temp_data = train_test_split(df, test_size=0.4, random_state=42)
valid_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Prepare data for XGBoost
dtrain = xgb.DMatrix(train_data[predictors], label=train_data[response])
dvalid = xgb.DMatrix(valid_data[predictors], label=valid_data[response])
dtest = xgb.DMatrix(test_data[predictors], label=test_data[response])

In [ ]:
# Step 3: Hyperparameter tuning using validation set
param_grid = {
    'max_depth': [3, 6, 9],
    'eta': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

best_params = None
best_rmse = float("inf")

for max_depth in param_grid['max_depth']:
    for eta in param_grid['eta']:
        for subsample in param_grid['subsample']:
            for colsample_bytree in param_grid['colsample_bytree']:
                params = {
                    'objective': 'reg:squarederror',
                    'max_depth': max_depth,
                    'eta': eta,
                    'subsample': subsample,
                    'colsample_bytree': colsample_bytree,
                    'eval_metric': 'rmse'
                }
                model = xgb.train(
                    params, dtrain, num_boost_round=1000,
                    evals=[(dvalid, "Validation")],
                    early_stopping_rounds=10, verbose_eval=False
                )
                if model.best_score < best_rmse:
                    best_rmse = model.best_score
                    best_params = params

print("Best Parameters:", best_params)

In [ ]:
# Step 4: Train the final model with best parameters
best_params = {'objective': 'reg:squarederror', 'max_depth': 9, 'eta': 0.1, 'subsample': 1.0, 'colsample_bytree': 0.8, 'eval_metric': 'rmse'}

final_model = xgb.train(best_params, dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
preds = final_model.predict(dtest)


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(final_model, importance_type='weight', max_num_features=10, height=0.5)
plt.title("Variable Importance (Top 10)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
test_rmse = np.sqrt(mean_squared_error(test_data[response], preds))
test_mae = mean_absolute_error(test_data[response], preds)
test_r2 = r2_score(test_data[response], preds)


In [ ]:
# Output best parameters and test RMSE
print("Best Parameters:", best_params)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

In [ ]:
# Define a tolerance threshold
tolerance = 5  # Predictions within ±5 of actual values are considered accurate

# Calculate accuracy
accuracy = np.mean(np.abs(preds - test_data[response]) <= tolerance)

print(f"Test Accuracy: {accuracy * 100:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(final_model, importance_type='weight', max_num_features=20, height=0.5)
plt.title("Full Data Feature Importance (Top 20)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=6)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(final_model, importance_type='weight', max_num_features=37, height=0.5)
plt.title("Full Data Feature Importance (All 37)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=6)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Extract actual and predicted values for the entire test data
actual_values = test_data[response]
predicted_values = preds

# Scatter plot: Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(actual_values, predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(actual_values), min(predicted_values))
max_val = max(max(actual_values), max(predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("NOX Scatter Plot: Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply log transformation to actual and predicted values
log_actual_values = np.log1p(actual_values)  # log1p is log(x + 1), handles zero values
log_predicted_values = np.log1p(predicted_values)

# Scatter plot: Log-transformed Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(log_actual_values, log_predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(log_actual_values), min(log_predicted_values))
max_val = max(max(log_actual_values), max(log_predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Log(Actual Values + 1)")
plt.ylabel("Log(Predicted Values + 1)")
plt.title("NOX Scatter Plot: Log-Transformed Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
response = "no2"

# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(df[col].dtype, np.floating):
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values after conversion
df = df.dropna(subset=predictors + [response])

# Step 2: Split the data
train_data, temp_data = train_test_split(df, test_size=0.4, random_state=42)
valid_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Prepare data for XGBoost
dtrain = xgb.DMatrix(train_data[predictors], label=train_data[response])
dvalid = xgb.DMatrix(valid_data[predictors], label=valid_data[response])
dtest = xgb.DMatrix(test_data[predictors], label=test_data[response])

# Step 4: Train the final model with best parameters
best_params = {'objective': 'reg:squarederror', 'max_depth': 9, 'eta': 0.1, 'subsample': 1.0, 'colsample_bytree': 0.8, 'eval_metric': 'rmse'}

final_model = xgb.train(best_params, dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
preds = final_model.predict(dtest)

In [ ]:
import matplotlib.pyplot as plt

# Extract actual and predicted values for the entire test data
actual_values = test_data[response]
predicted_values = preds

# Scatter plot: Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(actual_values, predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(actual_values), min(predicted_values))
max_val = max(max(actual_values), max(predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("NO2 Scatter Plot: Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply log transformation to actual and predicted values
log_actual_values = np.log1p(actual_values)  # log1p is log(x + 1), handles zero values
log_predicted_values = np.log1p(predicted_values)

# Scatter plot: Log-transformed Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(log_actual_values, log_predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(log_actual_values), min(log_predicted_values))
max_val = max(max(log_actual_values), max(log_predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Log(Actual Values + 1)")
plt.ylabel("Log(Predicted Values + 1)")
plt.title("NO2 Scatter Plot: Log-Transformed Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import random

# Replace `test_data` and `preds` with your actual DataFrame and prediction array
# Ensure `test_data` contains the response variable and `preds` holds the model predictions

# Define response variable name
response = "nox"

# Take a random small sample of test data for visualization
sample_size = 500  # Adjust sample size as needed
sample_indices = random.sample(range(len(test_data)), sample_size)

# Extract actual and predicted values for the sample
actual_sample = test_data.iloc[sample_indices][response]
preds_sample = [preds[i] for i in sample_indices]

# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))
plt.scatter(actual_sample, preds_sample, alpha=0.5, edgecolors="k")

# Add a 45-degree line
min_val = min(min(actual_sample), min(preds_sample))
max_val = max(max(actual_sample), max(preds_sample))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Scatter Plot: Predicted vs Actual Values (500 Samples)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import random

# Define response variable name
response = "nox"

# Take a larger random sample of test data for visualization
sample_size = 1000  # Adjust sample size as needed
sample_indices = random.sample(range(len(test_data)), sample_size)

# Extract actual and predicted values for the sample
actual_sample = test_data.iloc[sample_indices][response]
preds_sample = [preds[i] for i in sample_indices]

# Scatter plot: Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(actual_sample, preds_sample, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(actual_sample), min(preds_sample))
max_val = max(max(actual_sample), max(preds_sample))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Scatter Plot: Predicted vs Actual Values (Larger Sample: 1000)")
plt.legend()
plt.grid(True)
plt.show()


*[Pasted model results redacted: metrics derived from confidential data are not kept in git.]*


# Subset Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# Subset the data
sound_low_temp = df[df["temp_f"] <= 75]
sound_low_temp_ws = sound_low_temp[sound_low_temp["wsp_ms"] <= 7]

# Renaming columns by replacing spaces with underscores
sound_low_temp_ws.rename(columns=lambda col: col.replace(" ", "_"), inplace=True)

# Define predictors and response
predictors = [
    "LAFmax", "LAFmin", "LCpeak", "LAeq", "LCeq", "LZeq_12.5Hz",
    "LZeq_16Hz", "LZeq_20Hz", "LZeq_25Hz", "LZeq_31.5Hz", "LZeq_40Hz",
    "LZeq_50Hz", "LZeq_63Hz", "LZeq_80Hz", "LZeq_100Hz", "LZeq_125Hz",
    "LZeq_160Hz", "LZeq_200Hz", "LZeq_250Hz", "LZeq_315Hz", "LZeq_400Hz",
    "LZeq_500Hz", "LZeq_630Hz", "LZeq_800Hz", "LZeq_1kHz", "LZeq_1.25kHz",
    "LZeq_1.6kHz", "LZeq_2kHz", "LZeq_2.5kHz", "LZeq_3.15kHz", "LZeq_4kHz",
    "LZeq_5kHz", "LZeq_6.3kHz", "LZeq_8kHz", "LZeq_10kHz", "LZeq_12.5kHz",
    "LZeq_16kHz"
]
response = "nox"

# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(sound_low_temp_ws[col].dtype, np.floating):
        sound_low_temp_ws[col] = pd.to_numeric(sound_low_temp_ws[col], errors='coerce')

# Drop rows with missing values after conversion
sound_low_temp_ws = sound_low_temp_ws.dropna(subset=predictors + [response])

# Step 2: Split the data
subset_train_data, temp_data = train_test_split(sound_low_temp_ws, test_size=0.4, random_state=42)
subset_valid_data, subset_test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Prepare data for XGBoost
subset_dtrain = xgb.DMatrix(subset_train_data[predictors], label=subset_train_data[response])
subset_dvalid = xgb.DMatrix(subset_valid_data[predictors], label=subset_valid_data[response])
subset_dtest = xgb.DMatrix(subset_test_data[predictors], label=subset_test_data[response])

# Step 3: Hyperparameter tuning using validation set
param_grid = {
    'max_depth': [3, 6, 9],
    'eta': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

subset_best_params = None
subset_best_rmse = float("inf")



In [ ]:
for max_depth in param_grid['max_depth']:
    for eta in param_grid['eta']:
        for subsample in param_grid['subsample']:
            for colsample_bytree in param_grid['colsample_bytree']:
                params = {
                    'objective': 'reg:squarederror',
                    'max_depth': max_depth,
                    'eta': eta,
                    'subsample': subsample,
                    'colsample_bytree': colsample_bytree,
                    'eval_metric': 'rmse'
                }
                model = xgb.train(
                    params, subset_dtrain, num_boost_round=1000,
                    evals=[(subset_dvalid, "Validation")],
                    early_stopping_rounds=10, verbose_eval=False
                )
                if model.best_score < best_rmse:
                    subset_best_rmse = model.best_score
                    subset_best_params = params


# Print the best parameters and RMSE
print("Best Parameters:", subset_best_rmse)
print("Best Validation RMSE:", subset_best_params)

In [ ]:
# [Pasted model results redacted: metrics derived from confidential data are not kept in git.]


In [ ]:
subset_best_params = {'objective': 'reg:squarederror', 'max_depth': 9, 'eta': 0.1, 'subsample': 0.8, 'colsample_bytree': 1.0, 'eval_metric': 'rmse'}

subset_final_model = xgb.train(subset_best_params, subset_dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
subset_preds = subset_final_model.predict(subset_dtest)

subset_test_rmse = np.sqrt(mean_squared_error(subset_test_data[response], subset_preds))
subset_test_mae = mean_absolute_error(subset_test_data[response], subset_preds)
subset_test_r2 = r2_score(subset_test_data[response], subset_preds)

# Output best parameters and test RMSE
print("Best Parameters:", subset_best_params)
print("Test RMSE:", subset_test_rmse)
print("Test MAE:", subset_test_mae)
print("Test R2:", subset_test_r2)

In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(subset_final_model, importance_type='weight', max_num_features=20, height=0.5)
plt.title("Subset Model Feature Importance (Top 20)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(subset_final_model, importance_type='weight', max_num_features=37, height=0.5)
plt.title("Subset Model Feature Importance (All 37)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=6)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract actual and predicted values for the entire test data
subset_actual_values = subset_test_data[response]
subset_predicted_values = subset_preds

# Scatter plot: Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(subset_actual_values, subset_predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(subset_actual_values), min(subset_predicted_values))
max_val = max(max(subset_actual_values), max(subset_predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Subset NOx Scatter Plot: Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Apply log transformation to actual and predicted values
subset_log_actual_values = np.log1p(subset_actual_values)  # log1p is log(x + 1), handles zero values
subset_log_predicted_values = np.log1p(subset_predicted_values)

# Scatter plot: Log-transformed Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(subset_log_actual_values, subset_log_predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(subset_log_actual_values), min(subset_log_predicted_values))
max_val = max(max(subset_log_actual_values), max(subset_log_predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Log(Actual Values + 1)")
plt.ylabel("Log(Predicted Values + 1)")
plt.title("Subset NOx Scatter Plot: Log-Transformed Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()

## With Met

In [ ]:
# Get the column names of the DataFrame
column_names = sound_low_temp_ws.columns.tolist()
print(column_names)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Subset the data
sound_low_temp = df[df["temp_f"] <= 75]
sound_low_temp_ws = sound_low_temp[sound_low_temp["wsp_ms"] <= 7]

# Renaming columns by replacing spaces with underscores
sound_low_temp_ws.rename(columns=lambda col: col.replace(" ", "_"), inplace=True)

# Define predictors (including meteorological variables)
predictors = [
    "LAFmax", "LAFmin", "LCpeak", "LAeq", "LCeq", "LZeq_12.5Hz",
    "LZeq_16Hz", "LZeq_20Hz", "LZeq_25Hz", "LZeq_31.5Hz", "LZeq_40Hz",
    "LZeq_50Hz", "LZeq_63Hz", "LZeq_80Hz", "LZeq_100Hz", "LZeq_125Hz",
    "LZeq_160Hz", "LZeq_200Hz", "LZeq_250Hz", "LZeq_315Hz", "LZeq_400Hz",
    "LZeq_500Hz", "LZeq_630Hz", "LZeq_800Hz", "LZeq_1kHz", "LZeq_1.25kHz",
    "LZeq_1.6kHz", "LZeq_2kHz", "LZeq_2.5kHz", "LZeq_3.15kHz", "LZeq_4kHz",
    "LZeq_5kHz", "LZeq_6.3kHz", "LZeq_8kHz", "LZeq_10kHz", "LZeq_12.5kHz",
    "LZeq_16kHz",
    # Meteorological variables
    "temp_f", "wsp_ms", "wdr_deg", "relh_percent", "pressure_altcorr", "water_vapor_mr_g_kg", "solr_W_m2"
]
response = "nox"

# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(sound_low_temp_ws[col].dtype, np.floating):
        sound_low_temp_ws[col] = pd.to_numeric(sound_low_temp_ws[col], errors='coerce')

# Drop rows with missing values after conversion
sound_low_temp_ws = sound_low_temp_ws.dropna(subset=predictors + [response])

# Step 2: Split the data
met_subset_train_data, temp_data = train_test_split(sound_low_temp_ws, test_size=0.4, random_state=42)
met_subset_valid_data, met_subset_test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Prepare data for XGBoost
met_subset_dtrain = xgb.DMatrix(met_subset_train_data[predictors], label=met_subset_train_data[response])
met_subset_dvalid = xgb.DMatrix(met_subset_valid_data[predictors], label=met_subset_valid_data[response])
met_subset_dtest = xgb.DMatrix(met_subset_test_data[predictors], label=met_subset_test_data[response])

# Step 3: Hyperparameter tuning using validation set
param_grid = {
    'max_depth': [3, 6, 9],
    'eta': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

met_subset_best_params = None
met_subset_best_rmse = float("inf")




In [ ]:
for max_depth in param_grid['max_depth']:
    for eta in param_grid['eta']:
        for subsample in param_grid['subsample']:
            for colsample_bytree in param_grid['colsample_bytree']:
                params = {
                    'objective': 'reg:squarederror',
                    'max_depth': max_depth,
                    'eta': eta,
                    'subsample': subsample,
                    'colsample_bytree': colsample_bytree,
                    'eval_metric': 'rmse'
                }
                model = xgb.train(
                    params, dtrain, num_boost_round=1000,
                    evals=[(dvalid, "Validation")],
                    early_stopping_rounds=10, verbose_eval=False
                )
                if model.best_score < best_rmse:
                    best_rmse = model.best_score
                    best_params = params


*[Pasted model results redacted: metrics derived from confidential data are not kept in git.]*


In [ ]:
met_subset_best_params = {'objective': 'reg:squarederror', 'max_depth': 9, 'eta': 0.1, 'subsample': 1.0, 'colsample_bytree': 0.8, 'eval_metric': 'rmse'}
met_subset_best_model = xgb.train(met_subset_best_params, met_subset_dtrain, num_boost_round=1000)



In [ ]:
# Step 5: Evaluate on test data
met_subset_preds = met_subset_best_model.predict(met_subset_dtest)
met_subset_test_rmse = np.sqrt(mean_squared_error(met_subset_test_data[response], met_subset_preds))
met_subset_test_r2 = r2_score(met_subset_test_data[response], met_subset_preds)

# Output results
print("Best Parameters:", subset_best_params)
print("Test RMSE:", subset_test_rmse)
print("Test MAE:", subset_test_mae)
print("Test R2:", subset_test_r2)


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(met_subset_best_model, importance_type='weight', max_num_features=20, height=0.5)
plt.title("Met Subset Feature Importance (Top 20)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(20, 40))  # Increase height for better spacing
xgb.plot_importance(met_subset_best_model, importance_type='weight', max_num_features=44, height=0.5)
plt.title("Met Subset Model Feature Importance (All 44)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract actual and predicted values for the entire test data
met_subset_actual_values = met_subset_test_data[response]
met_subset_predicted_values = met_subset_preds

# Scatter plot: Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(met_subset_actual_values, met_subset_predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(met_subset_actual_values), min(met_subset_predicted_values))
max_val = max(max(met_subset_actual_values), max(met_subset_predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Met Subset NOx Scatter Plot: Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Apply log transformation to actual and predicted values
met_subset_log_actual_values = np.log1p(met_subset_actual_values)  # log1p is log(x + 1), handles zero values
met_subset_log_predicted_values = np.log1p(met_subset_predicted_values)

# Scatter plot: Log-transformed Predicted vs Actual
plt.figure(figsize=(10, 8))
plt.scatter(met_subset_log_actual_values, met_subset_log_predicted_values, alpha=0.3, edgecolors="k", s=10)

# Add a 45-degree line
min_val = min(min(met_subset_log_actual_values), min(met_subset_log_predicted_values))
max_val = max(max(met_subset_log_actual_values), max(met_subset_log_predicted_values))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="45-degree line")

plt.xlabel("Log(Actual Values + 1)")
plt.ylabel("Log(Predicted Values + 1)")
plt.title("Met Subset NOx Scatter Plot: Log-Transformed Predicted vs Actual Values (Full Test Data)")
plt.legend()
plt.grid(True)
plt.show()

## Frequency corr > 0.9

In [ ]:
# Define predictors and response
predictors = [
    "LAFmax", "LAFmin", "LCpeak", "LAeq", "LCeq", "LZeq_12.5Hz",
    "LZeq_25Hz",
    "LZeq_63Hz", "LZeq_80Hz", "LZeq_100Hz", "LZeq_125Hz",
    "LZeq_160Hz", "LZeq_200Hz", "LZeq_250Hz",
    "LZeq_500Hz", "LZeq_1.25kHz",
    "LZeq_1.6kHz",
    "LZeq_5kHz", "LZeq_12.5kHz"# , "temp_f", "wsp_ms", "wdr_deg", "relh_percent", "pressure_altcorr", "rain", "solr_W_m2"
]
response = "nox"



In [ ]:
# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(df[col].dtype, np.floating):
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values after conversion
df = df.dropna(subset=predictors + [response])

# Step 2: Split the data
train_data, temp_data = train_test_split(df, test_size=0.4, random_state=42)
valid_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Prepare data for XGBoost
dtrain = xgb.DMatrix(train_data[predictors], label=train_data[response])
dvalid = xgb.DMatrix(valid_data[predictors], label=valid_data[response])
dtest = xgb.DMatrix(test_data[predictors], label=test_data[response])




In [ ]:
# Step 3: Hyperparameter tuning using validation set
param_grid = {
    'max_depth': [3, 6, 9],
    'eta': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

best_params = None
best_rmse = float("inf")

for max_depth in param_grid['max_depth']:
    for eta in param_grid['eta']:
        for subsample in param_grid['subsample']:
            for colsample_bytree in param_grid['colsample_bytree']:
                params = {
                    'objective': 'reg:squarederror',
                    'max_depth': max_depth,
                    'eta': eta,
                    'subsample': subsample,
                    'colsample_bytree': colsample_bytree,
                    'eval_metric': 'rmse'
                }
                model = xgb.train(
                    params, dtrain, num_boost_round=1000,
                    evals=[(dvalid, "Validation")],
                    early_stopping_rounds=10, verbose_eval=False
                )
                if model.best_score < best_rmse:
                    best_rmse = model.best_score
                    best_params = params
print("Best Parameters:", best_params)



In [ ]:
final_model = xgb.train(best_params, dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
preds = final_model.predict(dtest)

test_rmse = np.sqrt(mean_squared_error(test_data[response], preds))
test_mae = mean_absolute_error(test_data[response], preds)
test_r2 = r2_score(test_data[response], preds)

# Output best parameters and test RMSE
print("Best Parameters:", best_params)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

In [ ]:
best_params = {'objective': 'reg:squarederror', 'max_depth': 12, 'eta': 0.01, 'subsample': 0.8, 'colsample_bytree': 1.0, 'eval_metric': 'rmse'}
final_model = xgb.train(best_params, dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
preds = final_model.predict(dtest)

test_rmse = np.sqrt(mean_squared_error(test_data[response], preds))
test_mae = mean_absolute_error(test_data[response], preds)
test_r2 = r2_score(test_data[response], preds)

# Output best parameters and test RMSE
print("Best Parameters:", best_params)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

In [ ]:
best_params = {'objective': 'reg:squarederror', 'max_depth': 12, 'eta': 0.01, 'subsample': 0.6, 'colsample_bytree': 1.0, 'eval_metric': 'rmse'}
final_model = xgb.train(best_params, dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
preds = final_model.predict(dtest)

test_rmse = np.sqrt(mean_squared_error(test_data[response], preds))
test_mae = mean_absolute_error(test_data[response], preds)
test_r2 = r2_score(test_data[response], preds)

# Output best parameters and test RMSE
print("Best Parameters:", best_params)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted Y-axis spacing
plt.figure(figsize=(10, 20))  # Increase height for better spacing
xgb.plot_importance(final_model, importance_type='weight', max_num_features=10, height=0.5)
plt.title("Variable Importance (Top 10)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)  # Adjust the font size for readability
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()


In [ ]:
best_params = {'objective': 'reg:squarederror', 'max_depth': 12, 'eta': 0.01, 'subsample': 0.65, 'colsample_bytree': 1.0, 'eval_metric': 'rmse'}
final_model = xgb.train(best_params, dtrain, num_boost_round=1000)

# Step 5: Evaluate on test data
preds = final_model.predict(dtest)

test_rmse = np.sqrt(mean_squared_error(test_data[response], preds))
test_mae = mean_absolute_error(test_data[response], preds)
test_r2 = r2_score(test_data[response], preds)

# Output best parameters and test RMSE
print("Best Parameters:", best_params)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

*[Pasted model results redacted: metrics derived from confidential data are not kept in git.]*
